# Identifier Policy Checks

This notebook helps you inspect and debug BattINFO identifier rules.

Covers:
- canonical ID formats from policy
- current example IDs
- automated policy lint script


In [ ]:
from pathlib import Path
import json
import re
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # If notebook is opened from inside notebooks/, move to repo root.
    ROOT = ROOT.parent

assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('ROOT:', ROOT)


In [ ]:
policy = (ROOT / 'IDENTIFIER_POLICY.md').read_text(encoding='utf-8')
for marker in [
    'Canonical IRI regex',
    'Canonical vs Accepted Forms',
    'Short Human Identifier (ShortID)',
    'Resolver and Dereferencing Policy',
]:
    print(marker, '->', marker in policy)


In [ ]:
uid = r'[0-9a-hjkmnp-tv-z]{4}(?:-[0-9a-hjkmnp-tv-z]{4}){3}'
patterns = {
    'cell-type': re.compile(rf'^https://w3id\.org/battinfo/cell-type/{uid}$'),
    'cell': re.compile(rf'^https://w3id\.org/battinfo/cell/{uid}$'),
    'dataset': re.compile(rf'^https://w3id\.org/battinfo/dataset/{uid}$'),
}

samples = {
    'cell-type': 'https://w3id.org/battinfo/cell-type/7d9k-2m4p-8t3x-6nq5',
    'cell': 'https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8',
    'dataset': 'https://w3id.org/battinfo/dataset/1f8r-6v2k-9p4m-3t7x',
}
for k, v in samples.items():
    print(k, v, '->', bool(patterns[k].fullmatch(v)))


In [ ]:
example_files = [
    ROOT / 'assets' / 'examples' / 'cell-types' / 'A123__ANR26650M1-B.json',
    ROOT / 'assets' / 'examples' / 'cell-instances' / 'cell-3m6k-9t2p-7x4h-9nq8.json',
    ROOT / 'assets' / 'examples' / 'datasets' / 'dataset-1f8r-6v2k-9p4m-3t7x.json',
]
for f in example_files:
    doc = json.loads(f.read_text(encoding='utf-8'))
    if 'cell_type' in doc:
        print(f.name, 'cell_type.id =', doc['cell_type']['id'])
    elif 'cell_instance' in doc:
        print(f.name, 'cell_instance.id =', doc['cell_instance']['id'])
    elif 'dataset' in doc:
        print(f.name, 'dataset.id =', doc['dataset']['id'])


In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/lint_identifier_policy.py'],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
print('exit code:', result.returncode)


## Debug tip

If lint fails, start with the file path from output and inspect:
- `id` values
- `type_id` values
- `dataset_id` links
- `short_id` format
